In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

CSV_FILE = "Chocolate_sales.csv"
OUTPUT_DIR = "outputs"
PLOT_DIR = os.path.join(OUTPUT_DIR, "visualizations")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)


class SalesDataAnalyzer:
    """Menu-driven sales analysis using Pandas, NumPy and Matplotlib."""

    def __init__(self, file_path=CSV_FILE, auto_load=True):
        self.data = pd.DataFrame()
        self.file_path = ""
        self.current_plot = None
        self.last_result = None

        if auto_load:
            self.load_data(file_path)

    def __del__(self):
        try:
            plt.close("all")
        except Exception:
            pass

    def check_data(self):
        if self.data.empty:
            print("Please load a dataset first.")
            return False
        return True

    @staticmethod
    def ask_number(prompt, minimum=None, maximum=None, allow_float=True):
        while True:
            try:
                value = float(input(prompt)) if allow_float else int(input(prompt))

                if minimum is not None and value < minimum:
                    print(f"Enter a value greater than or equal to {minimum}.")
                    continue

                if maximum is not None and value > maximum:
                    print(f"Enter a value less than or equal to {maximum}.")
                    continue

                return value

            except ValueError:
                print("Invalid number. Please try again.")

    # ---------------- LOAD DATA ----------------

    def load_data(self, file_path):
        try:
            path = os.path.expanduser(file_path)

            if not os.path.isfile(path):
                print(f"File not found: {path}")
                return False

            if not path.lower().endswith(".csv"):
                print("Please select a CSV file.")
                return False

            loaded = pd.read_csv(path)
            loaded.columns = loaded.columns.astype(str).str.strip()

            if loaded.empty:
                print("The CSV file contains no records.")
                return False

            for column in loaded.columns:
                if column.strip().lower() in {
                    "date", "order date", "transaction date"
                }:
                    parsed = pd.to_datetime(
                        loaded[column],
                        errors="coerce",
                        dayfirst=True
                    )

                    if parsed.notna().any():
                        loaded[column] = parsed

            numeric_columns = {
                "amount", "sales", "revenue", "profit",
                "boxes shipped", "quantity", "price", "discount"
            }

            for column in loaded.columns:
                if column.strip().lower() in numeric_columns:
                    loaded[column] = pd.to_numeric(
                        loaded[column],
                        errors="coerce"
                    )

            self.data = loaded
            self.file_path = os.path.abspath(path)
            self.last_result = None

            print(
                f"Dataset loaded successfully: "
                f"{len(self.data)} rows, {len(self.data.columns)} columns."
            )
            print("Columns:", ", ".join(self.data.columns))

            return True

        except (
            OSError,
            pd.errors.ParserError,
            UnicodeError,
            ValueError
        ) as error:
            print(f"Unable to load dataset: {error}")
            return False

    # ---------------- EXPLORE DATA ----------------

    def explore_data(self):
        if not self.check_data():
            return

        while True:
            print("\n--- EXPLORE DATA ---")
            print("1. First 5 rows")
            print("2. Last 5 rows")
            print("3. Column names")
            print("4. Data types")
            print("5. Dataset information")
            print("6. Dataset shape")
            print("7. Descriptive summary")
            print("8. Missing values")
            print("9. Back")

            choice = input("Choice: ").strip()

            if choice == "1":
                print(self.data.head())
            elif choice == "2":
                print(self.data.tail())
            elif choice == "3":
                print(self.data.columns.tolist())
            elif choice == "4":
                print(self.data.dtypes)
            elif choice == "5":
                self.data.info()
            elif choice == "6":
                print(
                    f"Rows: {self.data.shape[0]}, "
                    f"Columns: {self.data.shape[1]}"
                )
            elif choice == "7":
                print(self.data.describe(include="all").T)
            elif choice == "8":
                print(self.data.isna().sum())
            elif choice == "9":
                return
            else:
                print("Invalid choice.")

    # ---------------- CLEAN DATA ----------------

    def clean_data(self):
        if not self.check_data():
            return

        while True:
            print("\n--- CLEAN DATA ---")
            print("1. Show missing values")
            print("2. Fill numeric missing values with mean")
            print("3. Fill missing values with a value")
            print("4. Drop rows with missing values")
            print("5. Remove duplicate rows")
            print("6. Convert column to numeric")
            print("7. Convert column to datetime")
            print("8. Back")

            choice = input("Choice: ").strip()

            try:
                if choice == "1":
                    print(self.data.isna().sum())

                elif choice == "2":
                    numeric = self.data.select_dtypes(
                        include=np.number
                    ).columns

                    if len(numeric) == 0:
                        print("No numeric columns found.")
                    else:
                        self.data[numeric] = self.data[numeric].fillna(
                            self.data[numeric].mean()
                        )
                        print("Numeric missing values filled with means.")

                elif choice == "3":
                    column = input("Column name: ").strip()

                    if column not in self.data:
                        print("Column not found.")
                        continue

                    value = input("Replacement value: ")

                    if pd.api.types.is_numeric_dtype(self.data[column]):
                        value = pd.to_numeric(value, errors="coerce")

                        if pd.isna(value):
                            print("Please enter a valid numeric value.")
                            continue

                    self.data[column] = self.data[column].fillna(value)
                    print("Missing values replaced.")

                elif choice == "4":
                    before = len(self.data)
                    self.data.dropna(inplace=True)
                    print(f"Dropped {before - len(self.data)} rows.")

                elif choice == "5":
                    before = len(self.data)
                    self.data.drop_duplicates(inplace=True)
                    print(f"Removed {before - len(self.data)} duplicate rows.")

                elif choice == "6":
                    column = input("Column name: ").strip()

                    if column not in self.data:
                        print("Column not found.")
                        continue

                    self.data[column] = pd.to_numeric(
                        self.data[column],
                        errors="coerce"
                    )
                    print("Column converted to numeric.")

                elif choice == "7":
                    column = input("Column name: ").strip()

                    if column not in self.data:
                        print("Column not found.")
                        continue

                    self.data[column] = pd.to_datetime(
                        self.data[column],
                        errors="coerce",
                        dayfirst=True
                    )
                    print("Column converted to datetime.")

                elif choice == "8":
                    return

                else:
                    print("Invalid choice.")

            except (ValueError, TypeError, KeyError) as error:
                print(f"Cleaning error: {error}")

    # ---------------- NUMPY OPERATIONS ----------------

    def get_numeric_array(self):
        numeric = self.data.select_dtypes(
            include=np.number
        ).columns.tolist()

        if not numeric:
            print("No numeric columns available.")
            return None

        print("Numeric columns:", numeric)
        column = input("Choose numeric column: ").strip()

        if column not in numeric:
            print("Invalid numeric column.")
            return None

        return self.data[column].dropna().to_numpy()

    def numpy_operations(self):
        if not self.check_data():
            return

        array = self.get_numeric_array()

        if array is None:
            return

        while True:
            print("\n--- NUMPY ARRAY OPERATIONS ---")
            print("1. Display array")
            print("2. Array properties")
            print("3. Indexing")
            print("4. Slicing")
            print("5. Reshape")
            print("6. Element-wise square")
            print("7. Back")

            choice = input("Choice: ").strip()

            try:
                if choice == "1":
                    print(array)

                elif choice == "2":
                    print("Dimensions:", array.ndim)
                    print("Shape:", array.shape)
                    print("Size:", array.size)
                    print("Data type:", array.dtype)

                elif choice == "3":
                    index = int(input("Index: "))
                    print(array[index])

                elif choice == "4":
                    start = int(input("Start index: "))
                    stop = int(input("Stop index: "))
                    step = int(input("Step: "))

                    if step == 0:
                        print("Step cannot be zero.")
                    else:
                        print(array[start:stop:step])

                elif choice == "5":
                    rows = int(input("Rows: "))
                    columns = int(input("Columns: "))
                    print(array.reshape(rows, columns))

                elif choice == "6":
                    print(np.square(array))

                elif choice == "7":
                    return

                else:
                    print("Invalid choice.")

            except (ValueError, IndexError) as error:
                print(f"NumPy operation error: {error}")

    # ---------------- MATHEMATICAL OPERATIONS ----------------

    def mathematical_operations(self):
        if not self.check_data():
            return

        array = self.get_numeric_array()

        if array is None:
            return

        print("\n1. Add")
        print("2. Subtract")
        print("3. Multiply")
        print("4. Divide")
        print("5. Square")
        print("6. Square root")
        print("7. Absolute value")

        choice = input("Choice: ").strip()

        try:
            if choice in {"1", "2", "3", "4"}:
                number = float(input("Number: "))

                if choice == "1":
                    result = array + number
                elif choice == "2":
                    result = array - number
                elif choice == "3":
                    result = array * number
                else:
                    if number == 0:
                        print("Division by zero is not allowed.")
                        return
                    result = array / number

                print(result)

            elif choice == "5":
                print(np.square(array))

            elif choice == "6":
                if np.any(array < 0):
                    print("Negative values exist; real square roots cannot be calculated.")
                else:
                    print(np.sqrt(array))

            elif choice == "7":
                print(np.abs(array))

            else:
                print("Invalid choice.")

        except (ValueError, TypeError, FloatingPointError) as error:
            print(f"Mathematical operation error: {error}")

    # ---------------- SEARCH / SORT / FILTER ----------------

    def search_sort_filter(self):
        if not self.check_data():
            return

        while True:
            print("\n--- SEARCH / SORT / FILTER ---")
            print("1. Search text/value")
            print("2. Sort rows")
            print("3. Filter exact value")
            print("4. Filter numeric range")
            print("5. Back")

            choice = input("Choice: ").strip()

            if choice == "5":
                return

            if choice not in {"1", "2", "3", "4"}:
                print("Invalid choice.")
                continue

            column = input("Column name: ").strip()

            if column not in self.data:
                print("Column not found.")
                continue

            try:
                if choice == "1":
                    value = input("Search text/value: ")

                    result = self.data[
                        self.data[column].astype(str).str.contains(
                            value,
                            case=False,
                            na=False,
                            regex=False
                        )
                    ]

                elif choice == "2":
                    order = input("Ascending? (y/n): ").strip().lower()

                    result = self.data.sort_values(
                        column,
                        ascending=order in {"y", "yes", "1"}
                    )

                elif choice == "3":
                    value = input("Exact value: ")

                    if pd.api.types.is_numeric_dtype(self.data[column]):
                        value = float(value)
                        result = self.data[self.data[column] == value]
                    else:
                        result = self.data[
                            self.data[column].astype(str).str.casefold()
                            == value.casefold()
                        ]

                else:
                    if not pd.api.types.is_numeric_dtype(self.data[column]):
                        print("Range filtering requires a numeric column.")
                        continue

                    low = float(input("Minimum value: "))
                    high = float(input("Maximum value: "))

                    if low > high:
                        low, high = high, low

                    result = self.data[
                        self.data[column].between(low, high)
                    ]

                self.last_result = result.copy()

                if result.empty:
                    print("No matching records.")
                else:
                    print(result.to_string(index=False))

                print(f"{len(result)} matching row(s).")

            except ValueError as error:
                print(f"Invalid filter input: {error}")

    # ---------------- COMBINE DATA ----------------

    def combine_data(self):
        if not self.check_data():
            return

        path = input("Path to second CSV: ").strip()

        if not os.path.isfile(path):
            print("File not found.")
            return

        try:
            other = pd.read_csv(path)
            other.columns = other.columns.astype(str).str.strip()

            print("\n1. Concatenate rows")
            print("2. Merge on key")
            print("3. Join on index")

            choice = input("Choice: ").strip()

            if choice == "1":
                result = pd.concat(
                    [self.data, other],
                    ignore_index=True,
                    sort=False
                )

            elif choice in {"2", "3"}:
                common = [
                    column for column in self.data.columns
                    if column in other.columns
                ]

                if not common:
                    print("No common columns to use as a key.")
                    return

                print("Common columns:", common)
                key = input("Key column: ").strip()

                if key not in common:
                    print("Invalid key.")
                    return

                if choice == "2":
                    result = pd.merge(
                        self.data,
                        other,
                        on=key,
                        how="inner",
                        suffixes=("_left", "_right")
                    )
                else:
                    result = self.data.set_index(key).join(
                        other.set_index(key),
                        how="inner",
                        lsuffix="_left",
                        rsuffix="_right"
                    ).reset_index()

            else:
                print("Invalid choice.")
                return

            print(result.head(10))

            if input("Use combined data? (y/n): ").strip().lower() in {
                "y", "yes"
            }:
                self.data = result
                print("Current dataset updated.")

        except (
            OSError,
            pd.errors.ParserError,
            ValueError,
            KeyError
        ) as error:
            print(f"Combine error: {error}")

    # ---------------- SPLIT DATA ----------------

    def split_data(self):
        if not self.check_data():
            return

        column = input("Split by column: ").strip()

        if column not in self.data:
            print("Column not found.")
            return

        try:
            for value, group in self.data.groupby(
                column,
                dropna=False
            ):
                safe = "".join(
                    character if character.isalnum() or character in "_-"
                    else "_"
                    for character in str(value)
                ).strip("_") or "missing"

                path = os.path.join(
                    OUTPUT_DIR,
                    f"split_{safe}.csv"
                )

                group.to_csv(path, index=False)
                print("Saved:", path)

            print("Data splitting completed.")

        except (OSError, ValueError, TypeError) as error:
            print(f"Split error: {error}")

    # ---------------- AGGREGATION ----------------

    def aggregate_functions(self):
        if not self.check_data():
            return

        numeric = self.data.select_dtypes(
            include=np.number
        ).columns.tolist()

        if not numeric:
            print("No numeric columns.")
            return

        print("Numeric columns:", numeric)
        column = input("Numeric column: ").strip()

        if column not in numeric:
            print("Invalid column.")
            return

        series = self.data[column]

        print(f"Sum: {series.sum()}")
        print(f"Mean: {series.mean()}")
        print(f"Median: {series.median()}")
        print(f"Min: {series.min()}")
        print(f"Max: {series.max()}")
        print(f"Count: {series.count()}")
        print(f"Std: {series.std()}")

        group = input("Group by column (Enter to skip): ").strip()

        if group:
            if group not in self.data:
                print("Column not found.")
                return

            grouped = self.data.groupby(
                group,
                dropna=False
            )[column]

            print(grouped.agg([
                "sum", "mean", "median", "min", "max", "count", "std"
            ]))

            print("\nGroup mean for each original row:")
            print(grouped.transform("mean"))

    # ---------------- STATISTICAL ANALYSIS ----------------

    def statistical_analysis(self):
        if not self.check_data():
            return

        numeric = self.data.select_dtypes(include=np.number)

        if numeric.empty:
            print("No numeric columns.")
            return

        print("\n--- DESCRIPTIVE STATISTICS ---")
        print(numeric.describe().T)
        print("\nVariance:\n", numeric.var())
        print("\nStandard deviation:\n", numeric.std())
        print("\nPercentiles:\n", numeric.quantile([0.25, 0.50, 0.75, 0.90]))
        print("\nCorrelation:\n", numeric.corr())

    # ---------------- PIVOT TABLE ----------------

    def create_pivot_table(self):
        if not self.check_data():
            return

        print("Columns:", self.data.columns.tolist())

        index = input("Index column: ").strip()
        values = input("Numeric values column: ").strip()
        columns = input("Columns field (optional): ").strip()

        if index not in self.data or values not in self.data:
            print("Invalid index or values column.")
            return

        if not pd.api.types.is_numeric_dtype(self.data[values]):
            print("Values column must be numeric.")
            return

        if columns and columns not in self.data:
            print("Columns field not found.")
            return

        try:
            pivot = pd.pivot_table(
                self.data,
                index=index,
                columns=columns or None,
                values=values,
                aggfunc="sum",
                fill_value=0
            )

            print(pivot)

            if input("Save pivot table to CSV? (y/n): ").strip().lower() in {
                "y", "yes"
            }:
                path = os.path.join(OUTPUT_DIR, "pivot_table.csv")
                pivot.to_csv(path)
                print("Saved:", path)

        except (ValueError, KeyError) as error:
            print(f"Pivot table error: {error}")

    # ---------------- REINDEX ----------------

    def reindex_dataset(self):
        if not self.check_data():
            return

        try:
            start = int(input("Starting index: "))
            self.data.index = range(start, start + len(self.data))
            print("Dataset re-indexed.")

        except ValueError as error:
            print(f"Reindex error: {error}")

    # ---------------- DESCRIPTIVE MENU ----------------

    def generate_descriptive_dataset(self):
        while True:
            print("\n--- DESCRIPTIVE / STATISTICAL ANALYSIS ---")
            print("1. Aggregating functions")
            print("2. Statistical analysis")
            print("3. Pivot table")
            print("4. Back")

            choice = input("Choice: ").strip()

            if choice == "1":
                self.aggregate_functions()
            elif choice == "2":
                self.statistical_analysis()
            elif choice == "3":
                self.create_pivot_table()
            elif choice == "4":
                return
            else:
                print("Invalid choice.")

    # ---------------- DATAFRAME / NUMPY MENU ----------------

    def perform_dataset_operations(self):
        while True:
            print("\n--- DATAFRAME / NUMPY OPERATIONS ---")
            print("1. NumPy array operations")
            print("2. Mathematical operations")
            print("3. Search / sort / filter")
            print("4. Combine DataFrames")
            print("5. Split DataFrame")
            print("6. Re-index dataset")
            print("7. Back")

            choice = input("Choice: ").strip()

            if choice == "1":
                self.numpy_operations()
            elif choice == "2":
                self.mathematical_operations()
            elif choice == "3":
                self.search_sort_filter()
            elif choice == "4":
                self.combine_data()
            elif choice == "5":
                self.split_data()
            elif choice == "6":
                self.reindex_dataset()
            elif choice == "7":
                return
            else:
                print("Invalid choice.")

    # ---------------- NUMERIC COLUMN ----------------

    def _numeric_column(self, prompt="Numeric column: "):
        columns = self.data.select_dtypes(
            include=np.number
        ).columns.tolist()

        if not columns:
            print("No numeric columns available.")
            return None

        print("Numeric columns:", columns)
        column = input(prompt).strip()

        if column not in columns:
            print("Please select a numeric column.")
            return None

        return column

    # ---------------- VISUALIZATION ----------------

    def visualize_data(self):
        if not self.check_data():
            return

        while True:
            print("\n--- DATA VISUALIZATION ---")
            print("1. Bar chart")
            print("2. Line chart")
            print("3. Scatter plot")
            print("4. Pie chart")
            print("5. Histogram")
            print("6. Stack plot")
            print("7. Correlation heatmap")
            print("8. Box plot")
            print("9. Multiple subplots")
            print("10. Back")

            choice = input("Choice: ").strip()

            if choice == "10":
                return

            try:
                numeric = self.data.select_dtypes(
                    include=np.number
                ).columns.tolist()

                # BAR CHART
                if choice == "1":
                    y = self._numeric_column()
                    category = input("Category column: ").strip()

                    if y is None or category not in self.data:
                        print("Invalid column.")
                        continue

                    values = self.data.groupby(
                        category,
                        dropna=False
                    )[y].sum().sort_values(ascending=False).head(20)

                    plt.figure(figsize=(10, 6))
                    plt.bar(
                        values.index.astype(str),
                        values.values,
                        color="steelblue",
                        alpha=0.85,
                        edgecolor="black",
                        label=y
                    )

                    plt.title(f"{y} by {category}")
                    plt.xlabel(category)
                    plt.ylabel(y)
                    plt.legend(loc="best")
                    plt.xticks(rotation=45)

                # LINE CHART
                elif choice == "2":
                    y = self._numeric_column()

                    if y is None:
                        continue

                    date_col = next(
                        (column for column in self.data.columns
                         if column.lower() == "date"),
                        None
                    )

                    x = input(
                        f"X-axis column (Enter for {date_col or 'row index'}): "
                    ).strip() or date_col

                    if x and x not in self.data:
                        print("Column not found.")
                        continue

                    plt.figure(figsize=(10, 6))

                    if x:
                        plot_data = self.data[[x, y]].copy()

                        if (
                            pd.api.types.is_datetime64_any_dtype(plot_data[x])
                            or pd.api.types.is_numeric_dtype(plot_data[x])
                        ):
                            plot_data = plot_data.groupby(
                                x,
                                dropna=False
                            )[y].sum().sort_index()
                        else:
                            plot_data = plot_data.groupby(
                                x,
                                dropna=False
                            )[y].sum()

                        plt.plot(
                            plot_data.index.astype(str),
                            plot_data.values,
                            color="darkorange",
                            marker="o",
                            linestyle="-",
                            linewidth=2,
                            label=y
                        )

                        plt.xlabel(x)

                    else:
                        plt.plot(
                            self.data[y].values,
                            color="darkorange",
                            marker="o",
                            linestyle="-",
                            linewidth=2,
                            label=y
                        )
                        plt.xlabel("Row")

                    plt.title(f"{y} Trend")
                    plt.ylabel(y)
                    plt.legend(loc="best")
                    plt.xticks(rotation=45)

                # SCATTER PLOT
                elif choice == "3":
                    x = self._numeric_column("X-axis numeric column: ")
                    y = self._numeric_column("Y-axis numeric column: ")

                    if x is None or y is None:
                        continue

                    plt.figure(figsize=(10, 6))
                    plt.scatter(
                        self.data[x],
                        self.data[y],
                        color="teal",
                        alpha=0.7,
                        marker="o",
                        label=f"{x} vs {y}"
                    )

                    plt.title(f"{x} vs {y}")
                    plt.xlabel(x)
                    plt.ylabel(y)
                    plt.legend(loc="best")

                # PIE CHART
                elif choice == "4":
                    category = input("Category column: ").strip()

                    if category not in self.data:
                        print("Column not found.")
                        continue

                    counts = self.data[category].value_counts(
                        dropna=False
                    ).head(10)

                    plt.figure(figsize=(8, 8))
                    plt.pie(
                        counts.values,
                        labels=counts.index.astype(str),
                        autopct="%1.1f%%",
                        startangle=90,
                        colors=plt.cm.Set3.colors,
                        wedgeprops={"edgecolor": "white"}
                    )

                    plt.title(f"{category} Distribution")
                    plt.legend(
                        counts.index.astype(str),
                        title=category,
                        loc="best"
                    )

                # HISTOGRAM
                elif choice == "5":
                    column = self._numeric_column()

                    if column is None:
                        continue

                    plt.figure(figsize=(10, 6))
                    plt.hist(
                        self.data[column].dropna(),
                        bins=20,
                        color="cornflowerblue",
                        alpha=0.8,
                        edgecolor="black",
                        label=column
                    )

                    plt.title(f"Distribution of {column}")
                    plt.xlabel(column)
                    plt.ylabel("Frequency")
                    plt.legend(loc="best")

                # STACK PLOT
                elif choice == "6":
                    category = input("Category/date column for x-axis: ").strip()

                    if category not in self.data:
                        print("Column not found.")
                        continue

                    if not numeric:
                        print("No numeric columns.")
                        continue

                    value = input(
                        f"Value column ({', '.join(numeric)}): "
                    ).strip()

                    if value not in numeric:
                        print("Invalid numeric column.")
                        continue

                    product = next(
                        (column for column in self.data.columns
                         if column.lower() == "product"),
                        None
                    )

                    group = input(
                        f"Stack/group column (Enter for {product or 'first category'}): "
                    ).strip() or product

                    if group is None or group not in self.data:
                        print("Group column not found.")
                        continue

                    pivot = self.data.pivot_table(
                        index=category,
                        columns=group,
                        values=value,
                        aggfunc="sum",
                        fill_value=0
                    )

                    pivot = pivot.loc[
                        :,
                        pivot.sum().sort_values(
                            ascending=False
                        ).head(8).index
                    ]

                    x_values = pivot.index.astype(str).tolist()
                    y_values = [
                        pivot[column].to_numpy()
                        for column in pivot.columns
                    ]

                    plt.figure(figsize=(12, 6))
                    plt.stackplot(
                        x_values,
                        *y_values,
                        labels=pivot.columns.astype(str),
                        alpha=0.8
                    )

                    plt.title(f"Stacked {value} by {category}")
                    plt.xlabel(category)
                    plt.ylabel(value)
                    plt.legend(
                        title=group,
                        loc="upper left",
                        bbox_to_anchor=(1.02, 1)
                    )
                    plt.xticks(rotation=45)

                # CORRELATION HEATMAP
                elif choice == "7":
                    if len(numeric) < 2:
                        print("At least two numeric columns are required.")
                        continue

                    plt.figure(figsize=(9, 7))
                    sns.heatmap(
                        self.data[numeric].corr(),
                        annot=True,
                        fmt=".2f",
                        cmap="coolwarm"
                    )

                    plt.title("Numeric Column Correlation Heatmap")

                # BOX PLOT
                elif choice == "8":
                    column = self._numeric_column()

                    if column is None:
                        continue

                    plt.figure(figsize=(8, 6))
                    plt.boxplot(
                        self.data[column].dropna(),
                        patch_artist=True,
                        boxprops={"facecolor": "lightblue"},
                        medianprops={"color": "red", "linewidth": 2}
                    )

                    plt.title(f"Box Plot of {column}")
                    plt.ylabel(column)
                    plt.legend([column], loc="best")

                # MULTIPLE SUBPLOTS
                elif choice == "9":
                    if not numeric:
                        print("No numeric columns.")
                        continue

                    selected = numeric[:4]

                    fig, axes = plt.subplots(
                        2,
                        2,
                        figsize=(12, 8)
                    )

                    for ax, column in zip(axes.flat, selected):
                        ax.hist(
                            self.data[column].dropna(),
                            bins=15,
                            color="skyblue",
                            alpha=0.8,
                            edgecolor="black",
                            label=column
                        )

                        ax.set_title(f"{column} distribution")
                        ax.set_xlabel(column)
                        ax.set_ylabel("Frequency")
                        ax.legend(loc="best")

                    for ax in list(axes.flat)[len(selected):]:
                        ax.set_visible(False)

                    fig.suptitle("Sales Data Overview")
                    plt.tight_layout()

                else:
                    print("Invalid choice.")
                    continue

                plt.tight_layout()
                self.current_plot = plt.gcf()

                plt.show(block=False)
                plt.pause(0.1)

                print("Visualization created. Use Save Visualization from the main menu.")

            except (
                ValueError,
                TypeError,
                KeyError,
                IndexError
            ) as error:
                print(f"Visualization error: {error}")

            except Exception as error:
                print(f"Unexpected visualization error: {error}")

    # ---------------- SAVE VISUALIZATION ----------------

    def save_visualization(self):
        if self.current_plot is None:
            print("Create a visualization first.")
            return

        filename = input("Filename (PNG/JPG): ").strip()

        if not filename:
            print("Filename cannot be empty.")
            return

        if not filename.lower().endswith((".png", ".jpg", ".jpeg")):
            filename += ".png"

        safe_name = os.path.basename(filename)
        path = os.path.join(PLOT_DIR, safe_name)

        try:
            self.current_plot.savefig(
                path,
                dpi=300,
                bbox_inches="tight"
            )

            print("Visualization saved:", path)

        except (OSError, ValueError) as error:
            print(f"Save visualization error: {error}")

    # ---------------- EXPORT DATASET ----------------

    def export_dataset(self):
        if not self.check_data():
            return

        filename = input("CSV filename (saved in outputs): ").strip()

        if not filename:
            print("Filename cannot be empty.")
            return

        if not filename.lower().endswith(".csv"):
            filename += ".csv"

        safe_name = os.path.basename(filename)
        path = os.path.join(OUTPUT_DIR, safe_name)

        try:
            self.data.to_csv(path, index=False)
            print("Dataset exported:", path)

        except OSError as error:
            print(f"Export error: {error}")

    # ---------------- EXPORT ANALYSIS ----------------

    def export_analysis(self):
        if not self.check_data():
            return

        try:
            numeric = self.data.select_dtypes(include=np.number)
            path = os.path.join(OUTPUT_DIR, "analysis_summary.txt")

            with open(path, "w", encoding="utf-8") as file:
                file.write("SALES DATA ANALYSIS SUMMARY\n")
                file.write("=" * 40 + "\n")
                file.write(f"Source: {self.file_path}\n")
                file.write(
                    f"Rows: {len(self.data)} | "
                    f"Columns: {len(self.data.columns)}\n\n"
                )

                file.write(
                    "Column names:\n"
                    + ", ".join(self.data.columns)
                    + "\n\n"
                )

                file.write(
                    "Missing values:\n"
                    + self.data.isna().sum().to_string()
                    + "\n\n"
                )

                if not numeric.empty:
                    file.write(
                        "Descriptive statistics:\n"
                        + numeric.describe().to_string()
                        + "\n"
                    )

                    file.write(
                        "\nTotal numeric sums:\n"
                        + numeric.sum().to_string()
                        + "\n"
                    )

            print("Analysis summary exported:", path)

        except OSError as error:
            print(f"Analysis export error: {error}")

    # ---------------- MAIN MENU ----------------

    def main_menu(self):
        while True:
            print("\n" + "=" * 60)
            print("PANDAS ANALYZER & DATA VISUALIZATION")
            print("=" * 60)

            print("1. Load dataset")
            print("2. Explore data")
            print("3. Clean data")
            print("4. DataFrame / NumPy operations")
            print("5. Descriptive statistics / pivot")
            print("6. Data visualization")
            print("7. Save visualization")
            print("8. Export dataset")
            print("9. Export analysis summary")
            print("10. Exit")

            choice = input("Enter choice: ").strip()

            if choice == "1":
                path = input(
                    f"CSV path (Enter for {CSV_FILE}): "
                ).strip() or CSV_FILE

                self.load_data(path)

            elif choice == "2":
                self.explore_data()

            elif choice == "3":
                self.clean_data()

            elif choice == "4":
                self.perform_dataset_operations()

            elif choice == "5":
                self.generate_descriptive_dataset()

            elif choice == "6":
                self.visualize_data()

            elif choice == "7":
                self.save_visualization()

            elif choice == "8":
                self.export_dataset()

            elif choice == "9":
                self.export_analysis()

            elif choice == "10":
                print("Exiting the program. Goodbye!")
                break

            else:
                print("Invalid choice. Enter a number from 1 to 10.")


def main():
    print("=" * 60)
    print("WELCOME TO PANDAS ANALYZER")
    print("=" * 60)

    analyzer = SalesDataAnalyzer()
    analyzer.main_menu()


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\nProgram stopped by user.")